In [7]:
# --- Parameter sweep → min/max params per model (aligned to your CLI) ---
import os, itertools
from typing import Dict, Any, Iterable, List
import pandas as pd
import torch
from hydra import compose
%load_ext autoreload
%autoreload 2
import networkx as nx
import numpy as np
import csv
import os
import os.path as osp
from hydra import compose, initialize
from hydra.utils import instantiate
from hydra.core.global_hydra import GlobalHydra  # Import GlobalHydra explicitly
from ogbench.utils.config_resolvers import (
    get_default_transform,
    get_monitor_metric,
    get_monitor_mode,
    infer_in_channels,
)
import pandas as pd

# stay off GPU
os.environ["CUDA_VISIBLE_DEVICES"] = ""
torch.set_grad_enabled(False)

PROJECT_CFG_NAME = "train.yaml"

# -------------------------
# Grids (matching your CLI)
# -------------------------
DATASETS = ["covidaki", "motrpac", "addneuromed", "parkinsons"]
ADJ_THRESHOLDS = [0.8, 0.85]  # from your example; add more if you want
DATALOADER_BATCH_SIZES = [8, 16]
NODE_SAMPLE_RATIOS = [1.0, 0.5, 0.2, 0.125, "full"]
SAMPLE_METHODS = ["variance", "random", "correlation"]

OPT_LRS = [0.001]
OPT_WD = [0.0004]  # your working config uses 0.0004

FE_OUT = [64, 128, 256]          # adjust if you want [32, 64, 128] like the example
#FE_PROJ_DROPOUT = [0.0, 0.25]

BB_NUM_LAYERS = [2, 4]
BB_DROPOUT = [0.2, 0.4]    # skip for chebnet
BB_ACT = ["relu"]          # skip for chebnet

READOUT_POOL = ["mean", "sum"]

# Models (use the exact names from your CLI)
MODEL_KEYS = [
    "sagn",
    "chebnet",
    "mlp",
    "gin",
    "gatv4",
    "gcn",
    "gat",
    "gatv2",
    "graph_sage",
]

# Model-specific grids (only things you actually have)
PER_MODEL_GRID: Dict[str, Dict[str, Iterable[Any]]] = {
    "gcn": {
        "model.backbone.num_layers": [2, 4],},
    "gat": {
        "model.backbone.heads": [2, 4],
        "model.backbone.num_layers": [2, 4],
        "model.backbone.v2": [False],
        # add if you truly have them:
        # "model.backbone.concat": [True],
    },
    "gin": {
        "model.backbone.num_layers": [2, 4],
    },
    "gatv2": {
        "model.backbone.v2": [True],
        "model.backbone.heads": [2, 4],
        "model.backbone.num_layers": [2, 4],
    },
    "gatv4": {
        "model.backbone.hidden_channels": [[8, 16], [64, 128]],
        "model.backbone.heads": [[3, 3]],
    },
    "graph_sage": {
        "model.backbone.num_layers": [2, 4],
    },
    "chebnet": {
        "model.backbone.K": [2, 3],
        "model.backbone.num_layers": [2, 4],
    },
    "mlp": {
        "model.backbone.hidden_channels": [
            [128, 64, 32],
            [512, 256, 128],
            [1024, 512, 256],
        ],
    },
    "sagn": {
        "model.backbone.hidden_channels": [128, 256],
        "model.backbone.dropout": [0.2, 0.4],
        "model.backbone.num_layers": [2, 4],
        "model.backbone.alpha": [0.5, 0.8],
    },
}

# -------------------------
# Helpers
# -------------------------
def count_trainable_params(model: torch.nn.Module) -> int:
    return sum(int(p.numel()) for p in model.parameters() if p.requires_grad)

def product_dict(grid: Dict[str, Iterable[Any]]) -> List[Dict[str, Any]]:
    keys = list(grid.keys())
    vals = [list(v) for v in grid.values()]
    return [{k: v for k, v in zip(keys, tup)} for tup in itertools.product(*vals)]

def to_override(k: str, v: Any) -> str:
    if isinstance(v, bool):
        return f"{k}={'true' if v else 'false'}"
    if isinstance(v, str):
        return f"{k}={v}"
    if isinstance(v, (list, tuple)):
        inner = ",".join(str(x) for x in v)
        return f"{k}=[{inner}]"
    return f"{k}={v}"

def build_overrides(base_overrides: List[str], hp_dict: Dict[str, Any]) -> List[str]:
    return base_overrides + [to_override(k, v) for k, v in hp_dict.items()]

# shared grid (keys exactly as in your CLI)
SHARED_GRID_BASE: Dict[str, Iterable[Any]] = {
    # "dataset": DATASETS,
    # "dataset.loader.parameters.adjacency_threshold": ADJ_THRESHOLDS,
    # "dataset.dataloader_params.batch_size": DATALOADER_BATCH_SIZES,
    # "dataset.loader.parameters.node_sample_ratio": NODE_SAMPLE_RATIOS,
    # "dataset.loader.parameters.method": SAMPLE_METHODS,

    "optimizer.parameters.lr": OPT_LRS,
    "optimizer.parameters.weight_decay": OPT_WD,

    "model.feature_encoder.out_channels": FE_OUT,
    #"model.feature_encoder.proj_dropout": FE_PROJ_DROPOUT,

    "model.readout.pooling_type": READOUT_POOL,
}

GLOBAL_BACKBONE = {
    #"model.backbone.dropout": BB_DROPOUT,
    "model.backbone.act": BB_ACT,
}

def merged_shared_grid_for_model(model_key: str) -> Dict[str, Iterable[Any]]:
    """
    Shared knobs (optimizer, encoder, readout) + 
    selective backbone defaults (skip some for chebnet).
    """
    g = dict(SHARED_GRID_BASE)

     # --- prune special cases ---
    if model_key == "sagn":
        g.pop("model.feature_encoder.out_channels", None)
    # Only add global backbone if the model isn't chebnet
    if model_key != "chebnet" and model_key != "sagn":
        g.update(GLOBAL_BACKBONE)

    # Always add the model-specific grid
    g.update(PER_MODEL_GRID.get(model_key, {}))
    if model_key == "gatv4" or model_key == "sagn" or model_key == "mlp":
        g.pop("model.feature_encoder.out_channels", None)
    return g

# -------------------------
# Sweep + count
# -------------------------
records = []
out_dir = "./stats/param_counts"
os.makedirs(out_dir, exist_ok=True)

# Clear GlobalHydra instance if already initialized
if GlobalHydra().is_initialized():
    GlobalHydra().clear()

initialize(config_path="../configs", job_name="job")

for model_key in MODEL_KEYS:
    shared_grid = merged_shared_grid_for_model(model_key)
    model_specific = PER_MODEL_GRID.get(model_key, {})
    full_grid = dict(shared_grid)
    full_grid.update(model_specific)

    for hp in product_dict(full_grid):
        overrides = [f"model={model_key}"]
        overrides = build_overrides(overrides, hp)

        try:
            cfg = compose(
                config_name=PROJECT_CFG_NAME,
                overrides=overrides,
                return_hydra_config=True,
            )
            model = instantiate(cfg.model, evaluator=cfg.evaluator, optimizer=cfg.optimizer, loss=cfg.loss).cpu()

            n_params = count_trainable_params(model)

            records.append({
                "model": model_key,
                "params": n_params,
                "overrides": " ".join(overrides),
                **hp,
            })
        except Exception as e:
            records.append({
                "model": model_key,
                "params": None,
                "overrides": " ".join(overrides),
                "error": str(e),
                **hp,
            })

df = pd.DataFrame(records)
os.makedirs(out_dir, exist_ok=True)
all_csv = os.path.join(out_dir, "params_all_combos.csv")
df.to_csv(all_csv, index=False)

def pick_row(group: pd.DataFrame, which: str):
    sub = group.dropna(subset=["params"])
    if sub.empty: return {"params": None, "overrides": None}
    idx = sub["params"].idxmin() if which == "min" else sub["params"].idxmax()
    row = sub.loc[idx]
    return {"params": int(row["params"]), "overrides": row["overrides"]}

summary_rows = []
for m, g in df.groupby("model", dropna=False):
    mn = pick_row(g, "min")
    mx = pick_row(g, "max")
    summary_rows.append({
        "model": m,
        "min_params": mn["params"],
        "min_overrides": mn["overrides"],
        "max_params": mx["params"],
        "max_overrides": mx["overrides"],
        "num_valid": int(g["params"].notna().sum()),
        "num_total": int(len(g)),
        "num_errors": int(g["params"].isna().sum()),
    })

summary = pd.DataFrame(summary_rows)
summary_csv = os.path.join(out_dir, "params_summary.csv")
summary.to_csv(summary_csv, index=False)

display(summary)
print(f"Saved:\n- {all_csv}\n- {summary_csv}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/tmp/ipykernel_1007740/556235106.py:187: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../configs", job_name="job")


,model,min_params,min_overrides,max_params,max_overrides,num_valid,num_total,num_errors
0,chebnet,21185,model=chebnet optimizer.parameters.lr=0.001 op...,855297,model=chebnet optimizer.parameters.lr=0.001 op...,24,24,0
1,gat,13249,model=gat optimizer.parameters.lr=0.001 optimi...,333057,model=gat optimizer.parameters.lr=0.001 optimi...,24,24,0
2,gatv2,21569,model=gatv2 optimizer.parameters.lr=0.001 opti...,596225,model=gatv2 optimizer.parameters.lr=0.001 opti...,24,24,0
3,gatv4,8139,model=gatv4 optimizer.parameters.lr=0.001 opti...,156147,model=gatv4 optimizer.parameters.lr=0.001 opti...,4,4,0
4,gcn,12993,model=gcn optimizer.parameters.lr=0.001 optimi...,331009,model=gcn optimizer.parameters.lr=0.001 optimi...,12,12,0
5,gin,21313,model=gin optimizer.parameters.lr=0.001 optimi...,594177,model=gin optimizer.parameters.lr=0.001 optimi...,12,12,0
6,graph_sage,21185,model=graph_sage optimizer.parameters.lr=0.001...,593153,model=graph_sage optimizer.parameters.lr=0.001...,12,12,0
7,mlp,356609,model=mlp optimizer.parameters.lr=0.001 optimi...,3426305,model=mlp optimizer.parameters.lr=0.001 optimi...,6,6,0
8,sagn,85377,model=sagn optimizer.parameters.lr=0.001 optim...,1263873,model=sagn optimizer.parameters.lr=0.001 optim...,32,32,0


Saved:
- ./stats/param_counts/params_all_combos.csv
- ./stats/param_counts/params_summary.csv
